# QA 3 (Unit 3): Naive Bayes Implementation
## Part 1: Manual Calculation + Part 2: Code Implementation

## Part 1: Manual Naive Bayes Calculation

### Dataset (Spam Detection)
| Message | Contains 'free' | Contains 'win' | Contains 'hello' | Label |
|---------|----------------|----------------|-----------------|-------|
| msg1    | Yes            | Yes            | No              | Spam  |
| msg2    | Yes            | No             | No              | Spam  |
| msg3    | No             | Yes            | No              | Spam  |
| msg4    | No             | No             | Yes             | Ham   |
| msg5    | No             | No             | Yes             | Ham   |
| msg6    | No             | No             | No              | Ham   |

### Manual Calculation for: ['free'=Yes, 'win'=Yes, 'hello'=No]

**Step 1 - Prior Probabilities:**
- P(Spam) = 3/6 = 0.5
- P(Ham)  = 3/6 = 0.5

**Step 2 - Likelihoods (with Laplace smoothing):**
- P(free=Yes | Spam) = (2+1)/(3+2) = 3/5 = 0.6
- P(win=Yes  | Spam) = (2+1)/(3+2) = 3/5 = 0.6
- P(hello=No | Spam) = (3+1)/(3+2) = 4/5 = 0.8
- P(free=Yes | Ham)  = (0+1)/(3+2) = 1/5 = 0.2
- P(win=Yes  | Ham)  = (0+1)/(3+2) = 1/5 = 0.2
- P(hello=No | Ham)  = (1+1)/(3+2) = 2/5 = 0.4

**Step 3 - Posterior (unnormalized):**
- P(Spam|msg) ∝ 0.5 × 0.6 × 0.6 × 0.8 = 0.144
- P(Ham|msg)  ∝ 0.5 × 0.2 × 0.2 × 0.4 = 0.008

**Prediction → SPAM** ✅

## Part 2: Code Implementation

In [ ]:
import numpy as np
import pandas as pd
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
print('Libraries imported successfully!')

In [ ]:
# ── Dataset ──────────────────────────────────────────────────
messages = [
    'free win prize money',
    'free offer click now',
    'win free cash today',
    'hello how are you',
    'meeting tomorrow morning',
    'see you later today',
    'free cash win now click',
    'lunch tomorrow with team',
]
labels = ['spam','spam','spam','ham','ham','ham','spam','ham']

df = pd.DataFrame({'message': messages, 'label': labels})
print(df)

In [ ]:
# ── Vectorize ─────────────────────────────────────────────────
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['message'])
y = df['label']

print('Feature names:', vectorizer.get_feature_names_out())
print('Shape:', X.shape)

In [ ]:
# ── Train / Test Split ────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

model = MultinomialNB()
model.fit(X_train, y_train)
print('Model trained!')

In [ ]:
# ── Predictions ───────────────────────────────────────────────
y_pred = model.predict(X_test)

print('Actual   :', list(y_test))
print('Predicted:', list(y_pred))
print('Accuracy :', round(accuracy_score(y_test, y_pred), 4))
print()
print(classification_report(y_test, y_pred))

In [ ]:
# ── Probabilities ─────────────────────────────────────────────
probs = model.predict_proba(X_test)
classes = model.classes_
prob_df = pd.DataFrame(probs, columns=classes)
prob_df['Predicted'] = y_pred
prob_df['Actual'] = list(y_test)
print(prob_df.round(4))

In [ ]:
# ── Manual Naive Bayes from Scratch ──────────────────────────
class ManualNaiveBayes:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.priors = {}
        self.likelihoods = {}
        for c in self.classes:
            X_c = X[y == c]
            self.priors[c] = len(X_c) / len(X)
            self.likelihoods[c] = (X_c.sum(axis=0) + 1) / (X_c.sum() + X.shape[1])

    def predict(self, X):
        preds = []
        for x in X:
            scores = {}
            for c in self.classes:
                score = np.log(self.priors[c])
                score += np.sum(np.log(self.likelihoods[c]) * x)
                scores[c] = score
            preds.append(max(scores, key=scores.get))
        return preds

X_dense = X.toarray()
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_dense, np.array(labels), test_size=0.3, random_state=42)

manual_nb = ManualNaiveBayes()
manual_nb.fit(X_train_d, y_train_d)
manual_preds = manual_nb.predict(X_test_d)

print('Manual NB Predictions:', manual_preds)
print('Sklearn Predictions  :', list(y_pred))
print('Manual Accuracy:', round(accuracy_score(y_test_d, manual_preds), 4))

In [ ]:
# ── Compare Manual vs Sklearn ─────────────────────────────────
compare_df = pd.DataFrame({
    'Actual'      : list(y_test_d),
    'Manual NB'   : manual_preds,
    'Sklearn NB'  : list(y_pred),
    'Match'       : [m == s for m, s in zip(manual_preds, list(y_pred))]
})
print(compare_df)
print()
print('Manual  Accuracy:', round(accuracy_score(y_test_d, manual_preds), 4))
print('Sklearn Accuracy:', round(accuracy_score(y_test, y_pred), 4))

## Summary

**How Naive Bayes Works:**
- Based on Bayes theorem: P(class|features) ∝ P(class) × P(features|class)
- Assumes all features are independent (naive assumption)

**Prior:** P(Spam) = number of spam / total messages

**Likelihood:** P(word | class) = count of word in class / total words in class

**Prediction:** Choose the class with the highest posterior probability